# FEC Campaign Finance Database - Quick Start

Query the FEC campaign finance database directly from Colab. No download needed — DuckDB fetches only the pages your query touches via HTTP range requests.

**Data source:** [FEC Bulk Data](https://www.fec.gov/data/browse-data/?tab=bulk-data) | **GitHub:** [ian-nason/fec-database](https://github.com/ian-nason/fec-database)

In [ ]:
!pip install -q duckdb

import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("""
    ATTACH 'https://huggingface.co/datasets/Nason/fec-database/resolve/main/fec.duckdb'
    AS fec (READ_ONLY)
""")
con.sql("SELECT * FROM fec._metadata ORDER BY row_count DESC").show()

## Top Presidential Candidates by Individual Donations (2024)

In [ ]:
df = con.sql("""
    SELECT c.CAND_NAME AS candidate, c.CAND_PTY_AFFILIATION AS party,
        SUM(i.TRANSACTION_AMT) AS total_raised,
        COUNT(*) AS num_donations
    FROM fec.individual_contributions i
    JOIN fec.candidates c ON i.CMTE_ID = c.CAND_PCC AND i.cycle = c.cycle
    WHERE i.cycle = 2024 AND c.CAND_OFFICE = 'P' AND i.TRANSACTION_AMT > 0
    GROUP BY 1, 2 ORDER BY total_raised DESC LIMIT 15
""").df()

df.plot.barh(x='candidate', y='total_raised', figsize=(10, 6), legend=False)
import matplotlib.pyplot as plt
plt.xlabel('Total Raised ($)')
plt.title('Top Presidential Candidates by Individual Donations (2024)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
df

## Monthly Donation Trend

In [ ]:
df = con.sql("""
    SELECT DATE_TRUNC('month', TRANSACTION_DT) AS month,
        COUNT(*) AS donations,
        SUM(TRANSACTION_AMT) AS total
    FROM fec.individual_contributions
    WHERE cycle = 2024 AND TRANSACTION_DT IS NOT NULL
        AND TRANSACTION_AMT > 0
        AND TRANSACTION_DT BETWEEN '2023-01-01' AND '2024-12-31'
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(df['month'], df['total'] / 1e6, alpha=0.7, label='Total $ (millions)')
ax1.set_ylabel('Total Donations ($M)')
ax2 = ax1.twinx()
ax2.plot(df['month'], df['donations'] / 1000, color='red', label='# Donations (thousands)')
ax2.set_ylabel('Number of Donations (K)')
plt.title('Monthly Individual Donations (2024 Cycle)')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

## Top Donor Occupations

In [ ]:
df = con.sql("""
    SELECT OCCUPATION, COUNT(*) AS donations, SUM(TRANSACTION_AMT) AS total
    FROM fec.individual_contributions
    WHERE cycle = 2024 AND OCCUPATION IS NOT NULL
        AND OCCUPATION NOT IN ('RETIRED', 'NONE', 'N/A', 'NOT EMPLOYED', 'INFORMATION REQUESTED')
        AND TRANSACTION_AMT > 0
    GROUP BY 1 ORDER BY total DESC LIMIT 15
""").df()

df.plot.barh(x='OCCUPATION', y='total', figsize=(10, 6), legend=False)
plt.xlabel('Total Donated ($)')
plt.title('Top Donor Occupations (2024 Cycle)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Fundraising by Cycle Over Time

In [ ]:
df = con.sql("""
    SELECT cycle, COUNT(*) AS donations, SUM(TRANSACTION_AMT) AS total
    FROM fec.individual_contributions
    WHERE TRANSACTION_AMT > 0
    GROUP BY cycle ORDER BY cycle
""").df()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(df['cycle'], df['total'] / 1e9, alpha=0.7, color='steelblue')
ax1.set_ylabel('Total Individual Donations ($B)')
ax1.set_xlabel('Election Cycle')
plt.title('Individual Donations by Election Cycle')
plt.xticks(df['cycle'])
plt.tight_layout()
plt.show()